<a href="https://colab.research.google.com/github/bothwellishumba/Googlesheets/blob/main/Solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Background Overview of the Case**

This case involves analyzing trucking/load management data from the "Data" sheet in `Data.xlsx`. The dataset contains ~38 rows of shipment records with details like ID, Driver, Dispatcher, Dispatcher Team, PU/DO locations/dates, Notes (containing issue descriptions), Status, etc.

**Key business rules from the Instructions**:
- **Part 1**: Create an automated **Summary** table mirroring specific columns from the raw Data sheet (no manual entry).
- **Part 2**: Focus on loads with issues in the **Notes** column:
  - **BOL** issues → Calculate full days from Delivery Date (`DO Date`) to Feb 13, 2025.
  - **Awaiting Money** issues → Only count if ≥10 days from Delivery Date to Feb 13, 2025.
- Generate a **REPORT** sheet with a ranked summary by **Dispatcher Team** and **Dispatcher**: count of qualifying issue loads + total duration (days).

The goal is a scalable, formula-driven Google Sheets solution that updates automatically as data grows.

---

**Detailed Description of How I Solved the Case**

I first inspected the actual data using Python/pandas (in the sandbox environment) to understand column structure, date formats, and issue patterns in the **Notes** column. Issues like "Issue BOL...", "AWAITING MONEY...", etc., are embedded in Notes.

**Step 1: Part 1 – Automated Summary Table**
- Create a new sheet named `Summary`.
- In `Summary!A1`, use a dynamic `FILTER` or `QUERY` to pull data automatically.
- This ensures the table stays in sync with the "Data" sheet without manual copying.

**Step 2: Issue Detection & Duration Calculation (Helper in Data sheet)**
- Add a helper column (e.g., in column AA or a new sheet) to classify issues and compute durations using `ARRAYFORMULA`.

**Step 3: Part 2 – REPORT Sheet**
- Create a new sheet named `REPORT`.
- Use `QUERY` on the processed data for grouping, counting, and ranking.

All solutions are **fully formula-based** (no VBA/scripts needed) for easy maintenance in Google Sheets.

---

**Final REPORT Table** (as of the provided data, cutoff Feb 13, 2025)

| Dispatcher Team | Dispatcher         | Loads with Issues | Total Duration (days) |
|-----------------|--------------------|-------------------|-----------------------|
| Aris            | Liam Peterson      | 6                 | 71                    |
| Aris            | Nolan Pierce       | 6                 | 64                    |
| August          | Nathaniel Scott    | 4                 | 51                    |
| August          | Oscar Mitchell     | 3                 | 30                    |
| Samuel          | Elliott Drake      | 2                 | 28                    |
| August          | Matthew Hayes      | 2                 | 23                    |
| Aris            | Monica Rachels     | 1                 | 17                    |
| August          | Alexander Price    | 1                 | 17                    |
| August          | Ava Sanders        | 1                 | 17                    |
| August          | Kendrick Vaughn    | 1                 | 17                    |
| August          | Charles Holt       | 1                 | 15                    |
| Adrian          | James Harrison     | 1                 | 14                    |
| August          | Logan Barrett      | 1                 | 14                    |
| August          | Roland Sterling    | 1                 | 12                    |
| Aris            | Daniel Foster      | 1                 | 10                    |
| Aris            | Gregor Lennox      | 1                 | 10                    |
| August          | Henry Brooks       | 1                 | 10                    |
| August          | Noah Richardson    | 1                 | 10                    |
| August          | Oberon Vance       | 1                 | 10                    |
| Mitchell        | Ethan Rhodes       | 1                 | 10                    |

**Total loads with qualifying issues**: 20

---

**Exact Formulas Used**

### 1. Summary Sheet (A2)
```google-sheets
=FILTER(Data!A:AO, Data!A:A <> "")
```
(Or use `QUERY` for more control if column order needs remapping.)

### 2. Issue Classification + Duration (in Data sheet, e.g., new column AB2, then drag or use ARRAYFORMULA)
```google-sheets
=ARRAYFORMULA(
  IF(ROW()=1, "Issue Duration",
    IF(
      REGEXMATCH(UPPER(Data!AF2:AF), "BOL"),
      MAX(0, DATE(2025,2,13) - Data!N2:N),   // Column N = DO Date
      IF(
        REGEXMATCH(UPPER(Data!AF2:AF), "AWAITING MONEY"),
        IF(DATE(2025,2,13) - Data!N2:N >= 10,
           DATE(2025,2,13) - Data!N2:N,
           0),
        0
      )
    )
  )
)
```

### 3. REPORT Sheet (paste in A1)
```google-sheets
=QUERY(
  {Data!H:H, Data!G:G, IF(Data!AB:AB > 0, 1, 0), Data!AB:AB},
  "SELECT Col1, Col2, COUNT(Col3), SUM(Col4)
   WHERE Col3 = 1
   GROUP BY Col1, Col2
   ORDER BY SUM(Col4) DESC, COUNT(Col3) DESC
   LABEL Col1 'Team', Col2 'Dispatcher', COUNT(Col3) 'Loads with Issues', SUM(Col4) 'Total Duration (days)'",
  1
)
```

**Recommendations**:
- Format date columns properly.
- Freeze headers and apply conditional formatting.
- Protect formula sheets.
- This scales well for larger datasets.



**Bottom Line**:
The company is losing operational efficiency and likely money due to preventable paperwork and reconciliation delays. Focusing on BOL quality control and supporting the top dispatchers on the Aris team will deliver the fastest ROI.